In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("SHOW SCHEMAS").show(truncate=False)



+------------------+
|databaseName      |
+------------------+
|default           |
|information_schema|
+------------------+



In [0]:
spark.sql("USE SCHEMA default")



DataFrame[]

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable



In [0]:
events_df = spark.createDataFrame([
    ("s1", "u1", "click",  "2026-01-10 10:00:00"),
    ("s2", "u2", "view",   "2026-01-10 10:05:00"),
    ("s3", "u3", "buy",    "2026-01-10 10:10:00"),
    ("s4", "u1", "view",   "2026-01-10 10:15:00"),
], ["user_session", "user_id", "event_type", "event_time"]) \
.withColumn("event_time", to_timestamp("event_time"))


In [0]:
spark.sql("DROP TABLE IF EXISTS events_table_day5")

events_df.write.format("delta").mode("overwrite").saveAsTable("events_table_day5")
print("✅ Created Delta table events_table_day5")


✅ Created Delta table events_table_day5


In [0]:
spark.sql("SELECT * FROM events_table_day5 ORDER BY event_time").show(truncate=False)


+------------+-------+----------+-------------------+
|user_session|user_id|event_type|event_time         |
+------------+-------+----------+-------------------+
|s1          |u1     |click     |2026-01-10 10:00:00|
|s2          |u2     |view      |2026-01-10 10:05:00|
|s3          |u3     |buy       |2026-01-10 10:10:00|
|s4          |u1     |view      |2026-01-10 10:15:00|
+------------+-------+----------+-------------------+



In [0]:
## Incremental MERGE

In [0]:
updates_df = spark.createDataFrame([
    ("s2", "u2", "buy",  "2026-01-10 10:05:00"),  # update existing row
    ("s5", "u4", "click","2026-01-10 10:20:00")   # insert new row
], ["user_session", "user_id", "event_type", "event_time"]) \
.withColumn("event_time", to_timestamp("event_time"))


In [0]:
updates_df.createOrReplaceTempView("updates_view")


In [0]:
spark.sql("""
MERGE INTO events_table_day5 t
USING updates_view s
ON t.user_session = s.user_session AND t.event_time = s.event_time
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
print("✅ MERGE completed")


✅ MERGE completed


In [0]:
spark.sql("SELECT * FROM events_table_day5 ORDER BY event_time").show(truncate=False)


+------------+-------+----------+-------------------+
|user_session|user_id|event_type|event_time         |
+------------+-------+----------+-------------------+
|s1          |u1     |click     |2026-01-10 10:00:00|
|s2          |u2     |buy       |2026-01-10 10:05:00|
|s3          |u3     |buy       |2026-01-10 10:10:00|
|s4          |u1     |view      |2026-01-10 10:15:00|
|s5          |u4     |click     |2026-01-10 10:20:00|
+------------+-------+----------+-------------------+



In [0]:
## Time Travel

In [0]:
spark.sql("DESCRIBE HISTORY events_table_day5").show(truncate=False)


+-------+-------------------+--------------+-----------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+----------------------+-----------+-----------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
## Older version
spark.sql("SELECT * FROM events_table_day5 VERSION AS OF 0 ORDER BY event_time").show(truncate=False)


+------------+-------+----------+-------------------+
|user_session|user_id|event_type|event_time         |
+------------+-------+----------+-------------------+
|s1          |u1     |click     |2026-01-10 10:00:00|
|s2          |u2     |view      |2026-01-10 10:05:00|
|s3          |u3     |buy       |2026-01-10 10:10:00|
|s4          |u1     |view      |2026-01-10 10:15:00|
+------------+-------+----------+-------------------+



In [0]:
## Latest 
spark.sql("SELECT * FROM events_table_day5 ORDER BY event_time").show(truncate=False)


+------------+-------+----------+-------------------+
|user_session|user_id|event_type|event_time         |
+------------+-------+----------+-------------------+
|s1          |u1     |click     |2026-01-10 10:00:00|
|s2          |u2     |buy       |2026-01-10 10:05:00|
|s3          |u3     |buy       |2026-01-10 10:10:00|
|s4          |u1     |view      |2026-01-10 10:15:00|
|s5          |u4     |click     |2026-01-10 10:20:00|
+------------+-------+----------+-------------------+



In [0]:
## OPTIMIZE & ZORDER

In [0]:
spark.sql("""
OPTIMIZE events_table_day5
ZORDER BY (event_type, user_id)
""")
print("✅ OPTIMIZE done")


✅ OPTIMIZE done


In [0]:
## VACUUM cleanup

In [0]:
spark.sql("VACUUM events_table_day5 RETAIN 168 HOURS")
print("✅ VACUUM done")


✅ VACUUM done


In [0]:
## Final Outputs 

In [0]:
# FInal Table 
spark.sql("SELECT * FROM events_table_day5 ORDER BY event_time").show(truncate=False)

+------------+-------+----------+-------------------+
|user_session|user_id|event_type|event_time         |
+------------+-------+----------+-------------------+
|s1          |u1     |click     |2026-01-10 10:00:00|
|s2          |u2     |buy       |2026-01-10 10:05:00|
|s3          |u3     |buy       |2026-01-10 10:10:00|
|s4          |u1     |view      |2026-01-10 10:15:00|
|s5          |u4     |click     |2026-01-10 10:20:00|
+------------+-------+----------+-------------------+



In [0]:
# History 
spark.sql("DESCRIBE HISTORY events_table_day5").show(truncate=False)

+-------+-------------------+--------------+-----------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+----------------------+-----------+-----------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
# Time travel Version 0 
spark.sql("SELECT * FROM events_table_day5 VERSION AS OF 0 ORDER BY event_time").show(truncate=False)

+------------+-------+----------+-------------------+
|user_session|user_id|event_type|event_time         |
+------------+-------+----------+-------------------+
|s1          |u1     |click     |2026-01-10 10:00:00|
|s2          |u2     |view      |2026-01-10 10:05:00|
|s3          |u3     |buy       |2026-01-10 10:10:00|
|s4          |u1     |view      |2026-01-10 10:15:00|
+------------+-------+----------+-------------------+



In [0]:
# Show OPTIMIZE and VACUUM in history
spark.sql("DESCRIBE HISTORY events_table_day5").show(truncate=False)

+-------+-------------------+--------------+-----------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+----------------------+-----------+-----------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------